# Module 4 — Geographic Analysis

**Purpose:** Quantify territorial demand patterns, concentration, and growth disparities. Provides the geographic context for the CEO presentation and informs the cluster-to-geography cross-tab in Module 6.

**Inputs:** `monthly_territory_demand.parquet`, `outlet_sku_monthly_demand.parquet`, `base_sales_enriched.parquet`

---

## Analysis Roadmap

1. **Revenue Distribution & Concentration** — Pareto analysis by province, CR3, Gini coefficient
2. **Territory-Level Growth Dynamics** — YoY growth, ANOVA across regions
3. **Route & Outlet Density** — Outlet count vs. revenue correlation
4. **Payment Geography** — Cash vs. credit mix by region, chi-square test
5. **Export Results** — Summary tables and visualizations for downstream use

In [ ]:
import sys
from pathlib import Path

# PROJECT_ROOT = Path.cwd().parent.parent
PROJECT_ROOT = Path("../../..")
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import warnings
from scipy import stats
from scipy.stats import chi2_contingency
import os

warnings.filterwarnings('ignore')

# Set visualization defaults
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Import McKinsey theme
from src.viz.mckinsey_theme import (
    MCKINSEY_RC, BG_DARK, TEXT_WHITE, TEXT_DARK, GRID_COLOR,
    LINE_NOMINAL, LINE_VOLUME, LINE_REAL, HIGHLIGHT,
    create_mckinsey_figure, add_end_label
)

# Apply McKinsey theme
matplotlib.rcParams.update(MCKINSEY_RC)

print("Libraries imported successfully.")

## Section 1: Load Analytical Datasets

Load the pre-processed analytical datasets from Module 0. These datasets contain fully zero-filled, exhaustive primary keys.

In [ ]:
# Define data paths
intermediate_path = "../../../data/Intermediate"

# Load analytical datasets
print("Loading analytical datasets from Module 0...")

monthly_territory_demand = pd.read_parquet(os.path.join(intermediate_path, "monthly_territory_demand.parquet"))
outlet_sku_monthly_demand = pd.read_parquet(os.path.join(intermediate_path, "outlet_sku_monthly_demand.parquet"))
base_sales_enriched = pd.read_parquet(os.path.join(intermediate_path, "base_sales_enriched.parquet"))

print(f"✓ monthly_territory_demand: {monthly_territory_demand.shape}")
print(f"  Columns: {list(monthly_territory_demand.columns)[:5]}...")
print(f"\n✓ outlet_sku_monthly_demand: {outlet_sku_monthly_demand.shape}")
print(f"  Columns: {list(outlet_sku_monthly_demand.columns)[:8]}...")
print(f"\n✓ base_sales_enriched: {base_sales_enriched.shape}")
print(f"  Columns: {list(base_sales_enriched.columns)[:5]}...")

# Validate key columns exist
required_cols_territory = ['year_month', 'territory_id', 'gross_sales', 'gross_quantity', 'region', 'district', 'province']
required_cols_outlet_sku = ['outlet_id', 'item_code', 'year_month', 'gross_sales', 'gross_quantity', 'territory_id']
required_cols_base = ['outlet_id', 'territory_id', 'net_sales', 'quantity', 'payment_term', 'route_id']

assert all(col in monthly_territory_demand.columns for col in required_cols_territory), \
    f"Missing columns in monthly_territory_demand: {set(required_cols_territory) - set(monthly_territory_demand.columns)}"
assert all(col in outlet_sku_monthly_demand.columns for col in required_cols_outlet_sku), \
    f"Missing columns in outlet_sku_monthly_demand: {set(required_cols_outlet_sku) - set(outlet_sku_monthly_demand.columns)}"
assert all(col in base_sales_enriched.columns for col in required_cols_base), \
    f"Missing columns in base_sales_enriched: {set(required_cols_base) - set(base_sales_enriched.columns)}"

print("\n✓ All required columns validated.")

## Section 2: Revenue Distribution & Concentration

Aggregate gross sales and quantity by province, region, and district to identify revenue concentration and Pareto segments.

In [ ]:
# Aggregate gross_sales by province
province_revenue = monthly_territory_demand.groupby('province').agg({
    'gross_sales': 'sum',
    'gross_quantity': 'sum'
}).reset_index()

# Sort by gross_sales descending and compute cumulative share
province_revenue = province_revenue.sort_values('gross_sales', ascending=False).reset_index(drop=True)
province_revenue['revenue_share'] = province_revenue['gross_sales'] / province_revenue['gross_sales'].sum()
province_revenue['cumulative_revenue_share'] = province_revenue['revenue_share'].cumsum()
province_revenue['province_rank'] = range(1, len(province_revenue) + 1)

# CR3: Concentration Ratio of top 3
cr3 = province_revenue.head(3)['revenue_share'].sum()

# Summary statistics
print("=" * 80)
print("PROVINCE REVENUE CONTRIBUTION SUMMARY")
print("=" * 80)
print(f"\nTotal Provinces: {len(province_revenue)}")
print(f"Total Gross Sales: ${province_revenue['gross_sales'].sum():,.0f}")
print(f"CR3 (Top 3 Provinces): {cr3:.1%} {'(HIGH concentration risk)' if cr3 > 0.6 else '(moderate)'}")

# Top 5 Provinces
print(f"\nTop 5 Provinces by Revenue:")
top_5 = province_revenue.head(5)[['province', 'gross_sales', 'revenue_share']]
top_5['revenue_share_pct'] = (top_5['revenue_share'] * 100).round(2)
print(top_5[['province', 'gross_sales', 'revenue_share_pct']].to_string(index=False))

print("\n✓ Province revenue aggregation complete.")

In [ ]:
# Province Revenue Bar Chart - McKinsey Style
fig, ax = create_mckinsey_figure(title="Revenue Distribution by Province")

# Bar chart: revenue per province
bars = ax.barh(province_revenue['province'], province_revenue['gross_sales'], color=LINE_NOMINAL, alpha=0.85, 
               edgecolor=TEXT_WHITE, linewidth=1.5)

# Add value labels
for i, (idx, row) in enumerate(province_revenue.iterrows()):
    ax.text(row['gross_sales'] + province_revenue['gross_sales'].max() * 0.01, i, 
           f"${row['gross_sales']/1e6:.1f}M", va='center', fontsize=9, color=TEXT_WHITE, fontweight='bold')

ax.set_xlabel('Revenue ($)', fontsize=11, color=TEXT_WHITE)
ax.set_ylabel('Province', fontsize=11, color=TEXT_WHITE)
ax.set_xlim(0, province_revenue['gross_sales'].max() * 1.15)

plt.tight_layout()
plt.savefig('../../../data/output/m4_province_revenue.png', bbox_inches='tight', facecolor=BG_DARK, dpi=150)
plt.show()

print("✓ Province revenue chart generated and saved.")

## Section 3: Territory Concentration: CR3 and Gini Coefficient

Compute the concentration ratio and Gini coefficient for territory revenue, plot a Lorenz curve.

In [ ]:
# Aggregate gross_sales by territory
territory_revenue = monthly_territory_demand.groupby('territory_id').agg({
    'gross_sales': 'sum',
    'region': 'first',
    'province': 'first'
}).reset_index()

# Sort by gross_sales ascending for Lorenz curve
territory_revenue = territory_revenue.sort_values('gross_sales').reset_index(drop=True)
territory_revenue['cumulative_share'] = territory_revenue['gross_sales'].cumsum() / territory_revenue['gross_sales'].sum()
territory_revenue['equality_line'] = (territory_revenue.index + 1) / len(territory_revenue)

# Gini coefficient calculation
n = len(territory_revenue)
gini = (2 * (territory_revenue.index + 1).sum() * territory_revenue['gross_sales'].sum() - 
        (n + 1) * territory_revenue['gross_sales'].sum()) / (n * territory_revenue['gross_sales'].sum())

print("=" * 80)
print("GEOGRAPHIC CONCENTRATION ANALYSIS")
print("=" * 80)
print(f"\nTotal Territories: {len(territory_revenue)}")
print(f"Gini Coefficient: {gini:.3f} {'(HIGH concentration)' if gini > 0.5 else '(moderate)'}")
print(f"CR3: {cr3:.1%}")

print("\n✓ Territory concentration analysis complete.")

In [ ]:
# Lorenz Curve - McKinsey Style
fig, ax = create_mckinsey_figure(title="Lorenz Curve: Territory Revenue Distribution")

# Plot Lorenz curve
ax.plot(territory_revenue['equality_line'], territory_revenue['cumulative_share'], 
        color=LINE_NOMINAL, linewidth=3, marker='o', markersize=4, label='Lorenz Curve')

# Perfect equality line
ax.plot([0, 1], [0, 1], color=GRID_COLOR, linestyle='--', linewidth=2, alpha=0.7, label='Perfect Equality')

# Fill area between curves
ax.fill_between(territory_revenue['equality_line'], territory_revenue['cumulative_share'], 
                territory_revenue['equality_line'], alpha=0.3, color=LINE_REAL)

# Add Gini annotation
ax.text(0.7, 0.3, f'Gini = {gini:.3f}', fontsize=12, color=TEXT_WHITE, 
        bbox=dict(boxstyle='round,pad=0.4', facecolor=BG_DARK, edgecolor=TEXT_WHITE, linewidth=1))

ax.set_xlabel('Cumulative Share of Territories', fontsize=11, color=TEXT_WHITE)
ax.set_ylabel('Cumulative Share of Revenue', fontsize=11, color=TEXT_WHITE)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc='upper left', framealpha=0.0, fontsize=9)

plt.tight_layout()
plt.savefig('../../../data/output/m4_lorenz_curve.png', bbox_inches='tight', facecolor=BG_DARK, dpi=150)
plt.show()

print("✓ Lorenz curve generated and saved.")

## Section 4: Territory-Level Growth Dynamics

Compute YoY revenue growth per territory and run ANOVA across regions.

In [ ]:
# Prepare annual territory revenue
monthly_territory_demand['year'] = monthly_territory_demand['year_month'].str[:4].astype(int)
annual_territory_revenue = monthly_territory_demand.groupby(['territory_id', 'year']).agg({
    'gross_sales': 'sum',
    'region': 'first',
    'province': 'first'
}).reset_index()

# Compute YoY growth
annual_territory_revenue = annual_territory_revenue.sort_values(['territory_id', 'year'])
annual_territory_revenue['prev_year_sales'] = annual_territory_revenue.groupby('territory_id')['gross_sales'].shift(1)
annual_territory_revenue['yoy_growth'] = (annual_territory_revenue['gross_sales'] - annual_territory_revenue['prev_year_sales']) / annual_territory_revenue['prev_year_sales']

# Average YoY growth per territory
territory_growth = annual_territory_revenue.groupby('territory_id').agg({
    'yoy_growth': 'mean',
    'gross_sales': 'sum',
    'region': 'first',
    'province': 'first'
}).reset_index()

# Top and bottom 5 territories
top_5_growth = territory_growth.nlargest(5, 'yoy_growth')
bottom_5_growth = territory_growth.nsmallest(5, 'yoy_growth')

print("=" * 80)
print("TERRITORY GROWTH ANALYSIS")
print("=" * 80)
print(f"\nTop 5 Growing Territories:")
for _, row in top_5_growth.iterrows():
    print(f"  - {row['territory_id']} ({row['region']}): {row['yoy_growth']:.1%}")

print(f"\nBottom 5 Growing Territories:")
for _, row in bottom_5_growth.iterrows():
    print(f"  - {row['territory_id']} ({row['region']}): {row['yoy_growth']:.1%}")

# ANOVA: Mean sales differ across regions?
region_sales = [group['gross_sales'].values for name, group in annual_territory_revenue.groupby('region')]
f_stat, p_value = stats.f_oneway(*region_sales)

print(f"\n" + "=" * 80)
print("ANOVA: Regional Mean Sales Differences")
print("=" * 80)
print(f"H₀: Mean monthly gross_sales is equal across regions")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("✓ REJECT H₀ (p < 0.05): Regional means are significantly different.")
    print("  → Region-specific demand models are statistically justified.")
else:
    print("✓ FAIL TO REJECT H₀ (p ≥ 0.05): Regional means appear equal.")
    print("  → Use national-level demand model.")

print("\n✓ Territory growth analysis complete.")

In [ ]:
# Territory Growth Scatter Plot - McKinsey Style
fig, ax = create_mckinsey_figure(title="Territory Growth vs. Revenue Size")

# Scatter plot
ax.scatter(territory_growth['gross_sales'], territory_growth['yoy_growth'] * 100,
           s=territory_growth['gross_sales'] / 1e5, alpha=0.7, color=LINE_NOMINAL, 
           edgecolors=TEXT_WHITE, linewidth=1)

# Add labels for top/bottom
for _, row in pd.concat([top_5_growth, bottom_5_growth]).iterrows():
    ax.text(row['gross_sales'], row['yoy_growth'] * 100, row['territory_id'], 
           fontsize=8, color=TEXT_WHITE, ha='center')

ax.set_xlabel('Total Revenue ($)', fontsize=11, color=TEXT_WHITE)
ax.set_ylabel('Average YoY Growth (%)', fontsize=11, color=TEXT_WHITE)
ax.set_xlim(0, territory_growth['gross_sales'].max() * 1.1)
ax.set_ylim(territory_growth['yoy_growth'].min() * 100 * 1.2, territory_growth['yoy_growth'].max() * 100 * 1.2)

plt.tight_layout()
plt.savefig('../../../data/output/m4_territory_growth.png', bbox_inches='tight', facecolor=BG_DARK, dpi=150)
plt.show()

print("✓ Territory growth chart generated and saved.")

## Section 5: Payment Geography and Chi-Square Test

Analyze payment term mix by region and test for independence.

In [ ]:
# Payment term by region
payment_region = base_sales_enriched.groupby(['region', 'payment_term']).agg({
    'net_sales': 'sum',
    'quantity': 'sum',
    'document_number': 'count'
}).rename(columns={'document_number': 'n_invoices'}).reset_index()

# Pivot to contingency table
contingency_payment = payment_region.pivot(index='region', columns='payment_term', values='n_invoices').fillna(0)

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_payment)

print("=" * 80)
print("PAYMENT GEOGRAPHY ANALYSIS")
print("=" * 80)
print(f"\nPayment Term Distribution by Region:")
print(contingency_payment)

print(f"\n" + "=" * 80)
print("CHI-SQUARE TEST: Payment Term Independence from Region")
print("=" * 80)
print(f"H₀: Payment term is independent of region")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")

if p_value < 0.05:
    print("✓ REJECT H₀ (p < 0.05): Payment behavior is geographically segmented.")
    print("  → Region-specific credit risk and sales incentives required.")
else:
    print("✓ FAIL TO REJECT H₀ (p ≥ 0.05): Payment behavior is uniform across regions.")
    print("  → Apply national payment policies.")

print("\n✓ Payment geography analysis complete.")

In [ ]:
# Payment Mix by Region - McKinsey Style
fig, ax = create_mckinsey_figure(title="Payment Term Mix by Region")

# Normalize to percentages
payment_pct = contingency_payment.div(contingency_payment.sum(axis=1), axis=0) * 100

# Stacked bar chart
bottom = np.zeros(len(payment_pct))
colors_payment = [LINE_NOMINAL, LINE_VOLUME]
for i, term in enumerate(payment_pct.columns):
    ax.bar(payment_pct.index, payment_pct[term], bottom=bottom, label=term, 
           color=colors_payment[i % len(colors_payment)], alpha=0.85, edgecolor=GRID_COLOR, linewidth=0.5)
    bottom += payment_pct[term].values

ax.set_xlabel('Region', fontsize=11, color=TEXT_WHITE)
ax.set_ylabel('Share of Invoices (%)', fontsize=11, color=TEXT_WHITE)
ax.set_ylim(0, 105)
ax.legend(loc='upper left', framealpha=0.0, fontsize=9)

plt.tight_layout()
plt.savefig('../../../data/output/m4_payment_mix.png', bbox_inches='tight', facecolor=BG_DARK, dpi=150)
plt.show()

print("✓ Payment mix chart generated and saved.")

## Section 6: Export Geographic Analysis Results

Save all derived tables and generate a comprehensive module summary.

In [ ]:
# Export summary tables
output_path = "../../../data/output"
os.makedirs(output_path, exist_ok=True)

# 1. Province Revenue Summary
province_revenue_export = province_revenue[['province', 'gross_sales', 'gross_quantity', 'revenue_share', 
                                             'cumulative_revenue_share']].copy()
province_revenue_export.to_parquet(os.path.join(output_path, "m4_province_revenue.parquet"), index=False)
province_revenue_export.to_csv(os.path.join(output_path, "m4_province_revenue.csv"), index=False)

# 2. Territory Revenue and Growth
territory_growth_export = territory_growth[['territory_id', 'region', 'province', 'yoy_growth', 'gross_sales']].copy()
territory_growth_export.to_parquet(os.path.join(output_path, "m4_territory_growth.parquet"), index=False)
territory_growth_export.to_csv(os.path.join(output_path, "m4_territory_growth.csv"), index=False)

# 3. Payment Geography
payment_region_export = payment_region[['region', 'payment_term', 'net_sales', 'quantity', 'n_invoices']].copy()
payment_region_export.to_parquet(os.path.join(output_path, "m4_payment_region.parquet"), index=False)
payment_region_export.to_csv(os.path.join(output_path, "m4_payment_region.csv"), index=False)

print("=" * 80)
print("EXPORT SUMMARY")
print("=" * 80)
print(f"\n✓ Exported to {output_path}:")
print(f"  - m4_province_revenue.parquet/.csv ({len(province_revenue_export)} provinces)")
print(f"  - m4_territory_growth.parquet/.csv ({len(territory_growth_export)} territories)")
print(f"  - m4_payment_region.parquet/.csv ({len(payment_region_export)} region-payment combinations)")
print(f"\nAll files also saved as .csv for easy viewing.")

## Module 4 Conclusion

### Key Findings

**Geographic Concentration Risk**
- **CR3:** {cr3:.1%} → High/Moderate concentration
- **Gini Coefficient:** {gini:.3f} → High/Moderate concentration
- The top 3 provinces account for {cr3:.1%} of revenue, making the business dependent on few geographic areas.

**Territory Growth Disparities**
- Top growing territory: {top_5_growth.iloc[0]['territory_id']} ({top_5_growth.iloc[0]['yoy_growth']:.1%})
- Bottom growing territory: {bottom_5_growth.iloc[0]['territory_id']} ({bottom_5_growth.iloc[0]['yoy_growth']:.1%})
- ANOVA result: [Significant/Not significant] — region-specific models [are/are not] justified

**Payment Geography**
- Chi-square test: [Significant/Not significant] — payment behavior [varies/does not vary] by region
- [Notable regional differences in cash vs. credit noted]

### Downstream Implications

1. **Geographic Risk Management** — High CR3/Gini indicates supply chain vulnerability; diversify distribution or build redundancy in high-revenue provinces.
2. **Regional Targeting** — If ANOVA is significant, allocate forecasting effort and inventory differently by region.
3. **Credit Risk Segmentation** — If chi-square is significant, apply region-specific credit terms and collection strategies.
4. **CEO Presentation Context** — Use concentration metrics and growth maps to frame geographic strategy discussions.

---

In [ ]:
# Module Execution Metadata
print("=" * 80)
print("MODULE 4 EXECUTION SUMMARY")
print("=" * 80)
print(f"\nNotebook: m4_geographic_analysis.ipynb")
print(f"Module: Geographic Analysis")
print(f"Execution Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\nData Processed:")
print(f"  - monthly_territory_demand rows: {len(monthly_territory_demand):,}")
print(f"  - outlet_sku_monthly_demand rows: {len(outlet_sku_monthly_demand):,}")
print(f"  - base_sales_enriched rows: {len(base_sales_enriched):,}")
print(f"  - Unique territories: {monthly_territory_demand['territory_id'].nunique()}")
print(f"  - Unique regions: {monthly_territory_demand['region'].nunique()}")
print(f"  - Unique provinces: {monthly_territory_demand['province'].nunique()}")

print(f"\nAnalyses Completed:")
print(f"  ✓ Revenue Distribution & Concentration (Pareto)")
print(f"  ✓ Territory-Level Growth Dynamics (YoY, ANOVA)")
print(f"  ✓ Geographic Concentration (CR3, Gini, Lorenz)")
print(f"  ✓ Payment Geography (Chi-square)")
print(f"  ✓ Export of Summary Tables")

print(f"\nOutput Files Created:")
print(f"  ✓ m4_province_revenue.parquet/.csv")
print(f"  ✓ m4_territory_growth.parquet/.csv")
print(f"  ✓ m4_payment_region.parquet/.csv")
print(f"  ✓ m4_province_revenue.png")
print(f"  ✓ m4_lorenz_curve.png")
print(f"  ✓ m4_territory_growth.png")
print(f"  ✓ m4_payment_mix.png")

print("\n" + "=" * 80)
print("Module 4 Complete ✓")
print("=" * 80)